<a href="https://colab.research.google.com/github/johanndeboda/AAI2026/blob/2026fall/ex3_self_reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3 — Self-Reflection Prompt for Improving Output
**Tools:** Google Colab, Gemini API (google-genai SDK, gemini-2.5-flash, free tier)
**Method:** Draft summary → self-critique against explicit requirements → revised summary → automated checks (before vs after)
**Source:** fictional VoltCart Q3 customer support report

In [3]:
!pip install -q -U google-genai 2>/dev/null

In [4]:
from google import genai
from google.colab import userdata
import time, re, textwrap
import pandas as pd

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"

def ask(prompt):
    last_error = None
    for attempt in range(4):
        try:
            time.sleep(7)
            return client.interactions.create(model=MODEL, input=prompt).output_text.strip()
        except Exception as e:
            last_error = e
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                time.sleep(30)
                continue
            raise
    raise RuntimeError(f"Failed after 4 attempts. Last error: {last_error}")

def extract(tag, text):
    m = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.S)
    return m.group(1).strip() if m else ""

def show(title, text):
    print(f"\n--- {title} ---")
    for line in str(text).splitlines():
        print(textwrap.fill(line, width=110, subsequent_indent="    ") if line.strip() else "")

In [5]:
REPORT = """VoltCart Customer Support — Q3 2026 Report
Between July and September, the support team handled 18,400 tickets, up 22% from Q2. The increase was
driven mostly by late-delivery complaints after the company switched regional carriers in August;
late-delivery tickets rose from 2,100 in Q2 to 5,300 in Q3. Average first-response time grew from
4.2 hours to 7.8 hours, and customer satisfaction (CSAT) fell from 91% to 84%. Damaged-item tickets
stayed flat at about 1,900. Billing tickets dropped 15% after the new invoice page launched in July.
The team piloted an AI triage tool on 20% of tickets in September; on those tickets, first-response
time averaged 1.5 hours and CSAT was 89%. Two agents left in August and have not been replaced.
The report recommends expanding AI triage to all tickets in Q4 and renegotiating delivery terms with
the new carrier before the holiday season."""

REQUIREMENTS = """1. Audience: VP of Operations reading on a phone. Plain language, no support jargon.
2. Length: 80 words maximum.
3. Format: one headline sentence, then exactly 3 bullets, each starting with "- ".
4. Accuracy: use numbers exactly as written in the report. Add nothing that is not in the report.
5. Content: must cover ticket volume (18,400), response time (7.8 hours), CSAT (84%),
   the AI pilot result (1.5 hours, 89%), the main cause (carrier switch), and both recommended actions.
6. Tone: neutral and factual. No hype words."""

In [6]:
KEY_NUMBERS = ["18,400", "7.8", "84%", "1.5", "89%"]

def check(summary):
    words = len(summary.split())
    bullets = [l for l in summary.splitlines() if l.strip().startswith("- ")]
    missing = [n for n in KEY_NUMBERS if n not in summary]
    has_cause = "carrier" in summary.lower()
    has_recs = "triage" in summary.lower() and "renegotiat" in summary.lower()
    return {
        "Word count (max 80)": words,
        "Bullets (need 3)": len(bullets),
        "Missing key numbers": ", ".join(missing) if missing else "none",
        "Names cause (carrier)": has_cause,
        "Both recommendations": has_recs,
        "Meets all checks": words <= 80 and len(bullets) == 3 and not missing and has_cause and has_recs,
    }

## Step 1: Draft (before)
Baseline prompt with no audience, length, or format rules.

In [7]:
DRAFT_PROMPT = "Summarize this report.\n\n{report}"

draft = ask(DRAFT_PROMPT.format(report=REPORT))
show("DRAFT SUMMARY (before)", draft)
show("CHECKS", "\n".join(f"{k}: {v}" for k, v in check(draft).items()))


--- DRAFT SUMMARY (before) ---
The VoltCart Customer Support team handled 18,400 tickets in Q3 2026, a 22% increase from Q2. This surge was
    primarily driven by a rise in late-delivery complaints (from 2,100 to 5,300 tickets) after the company
    switched regional carriers in August. Consequently, average first-response time significantly increased
    from 4.2 to 7.8 hours, and customer satisfaction (CSAT) fell from 91% to 84%.

Despite these challenges, billing tickets dropped 15% following the launch of a new invoice page, and a pilot
    of an AI triage tool in September showed promise, achieving a 1.5-hour average first-response time and 89%
    CSAT on the 20% of tickets it handled. The team also experienced a staffing shortage with two agents
    leaving unreplaced.

The report recommends expanding the AI triage tool to all tickets in Q4 and renegotiating delivery terms with
    the new carrier before the holiday season.

--- CHECKS ---
Word count (max 80): 140
Bullets (nee

In [8]:
REFLECT_PROMPT = """You wrote the summary below. Review it critically, then improve it.

SOURCE REPORT:
<<<{report}>>>

YOUR SUMMARY:
<<<{summary}>>>

REQUIREMENTS:
{requirements}

Step 1 - Critique: for EACH numbered requirement, write PASS or FAIL and name the exact problem
(quote the words or count them). Also list any claim that is not supported by the source report.
Step 2 - Revise: write a new summary that fixes every FAIL. Do not add facts that are not in the report.
Count the words before finalizing.

Respond in exactly this format:
<critique>
1. PASS/FAIL - reason
...
Unsupported claims: none / list
</critique>
<revised>
the improved summary only
</revised>"""

In [9]:
def reflect_loop(summary, max_rounds=2):
    current = summary
    for r in range(1, max_rounds + 1):
        print(f"\n{'=' * 25} REFLECTION ROUND {r} {'=' * 25}")
        reply = ask(REFLECT_PROMPT.format(report=REPORT, summary=current, requirements=REQUIREMENTS))
        critique, revised = extract("critique", reply), extract("revised", reply)
        show("SELF-CRITIQUE", critique)
        show("REVISED SUMMARY", revised)
        results = check(revised)
        show("CHECKS", "\n".join(f"{k}: {v}" for k, v in results.items()))
        if not revised:
            print("No <revised> block found; keeping previous version.")
            continue
        current = revised
        if results["Meets all checks"]:
            print(f"\nAll checks passed after {r} round(s).")
            break
    return current

final = reflect_loop(draft)


========================= REFLECTION ROUND 1 =========================

--- SELF-CRITIQUE ---
1.  FAIL - "surge", "significantly increased", "challenges", "showed promise" are not plain, neutral language
    and can be considered hype words.
2.  FAIL - 147 words (exceeds 80-word maximum).
3.  FAIL - The summary uses three paragraphs instead of one headline sentence and exactly three bullets.
4.  PASS - All numbers used are exact and no information is added that is not in the source report.
5.  PASS - All required content elements are covered.
6.  FAIL - "surge", "significantly increased", "challenges", "showed promise" are not neutral and factual.

Unsupported claims: none

--- REVISED SUMMARY ---
Q3 2026 Customer Support metrics declined, with 18,400 tickets handled, average response time hitting 7.8
    hours, and CSAT falling to 84%.
- The main cause was late-delivery complaints, up from 2,100 to 5,300 after the August carrier switch.
- An AI pilot in September (20% of tickets) ach

In [10]:
show("BEFORE", draft)
show("AFTER", final)
pd.DataFrame({"Before": check(draft), "After": check(final)})


--- BEFORE ---
The VoltCart Customer Support team handled 18,400 tickets in Q3 2026, a 22% increase from Q2. This surge was
    primarily driven by a rise in late-delivery complaints (from 2,100 to 5,300 tickets) after the company
    switched regional carriers in August. Consequently, average first-response time significantly increased
    from 4.2 to 7.8 hours, and customer satisfaction (CSAT) fell from 91% to 84%.

Despite these challenges, billing tickets dropped 15% following the launch of a new invoice page, and a pilot
    of an AI triage tool in September showed promise, achieving a 1.5-hour average first-response time and 89%
    CSAT on the 20% of tickets it handled. The team also experienced a staffing shortage with two agents
    leaving unreplaced.

The report recommends expanding the AI triage tool to all tickets in Q4 and renegotiating delivery terms with
    the new carrier before the holiday season.

--- AFTER ---
Q3 2026 Customer Support metrics declined, with 18,400

,Before,After
Word count (max 80),140,69
Bullets (need 3),0,3
Missing key numbers,none,none
Names cause (carrier),True,True
Both recommendations,True,True
Meets all checks,False,True


## Iteration notes
- **Draft (before):** 140 words, 3 paragraphs, 0 bullets. All numbers were accurate, but it failed length,
  format, and tone ("surge", "significantly increased", "showed promise").
- **Self-critique flagged:** 4 FAILs (audience/plain language, length, format, tone); accuracy and content passed.
- **After:** 69 words, headline + 3 bullets, all key numbers, cause, and both recommendations. Passed all checks in 1 round.
- **Observation:** the model's critique counted 147 words; the actual count was 140. Self-assessment isn't
  fully reliable, so the automated checks in Cell 5 verify the improvement independently.
- **Trade-off:** to fit 80 words, the revision dropped secondary details (22% growth, billing drop, staffing).
  That's appropriate for a VP skim, but the "expand AI triage" bullet also lost "to all tickets."
- **Takeaway:** explicit, checkable criteria turned a generic summary into one fit for a specific reader.